# Cyrus, Winnowed

In [1]:
from winnow_fcns import *
from sim_utils import *
import numpy as np
from pathlib import Path
import os
import csv

In [2]:
# rng = np.random.default_rng(seed=42)
# rng2 = np.random.default_rng(seed=180)
# N = 30
# std = 0.1
# ber = 0.25


# init_key = rng2.integers(low=0, high=2, size=N)


# print(init_key)
# key = simulate_error(init_key, rng, ber=ber, N=N)
# print(key)


#------------------files---------------------------------------------
current_file = Path.cwd()

init_file = current_file.parent / "initial_cyrus_bits.csv"
ch1_file = current_file.parent / "ch1_cyrus.csv"
ch2_file = current_file.parent / "ch2_cyrus.csv"



def load_data(init_file, ch1_file, ch2_file):
    init = np.loadtxt(init_file, delimiter=',', dtype=int)
    ch1 = np.loadtxt(ch1_file, delimiter=',')
    ch2 = np.loadtxt(ch2_file, delimiter=',')
    return init, ch1, ch2


init_key, ch1, ch2 = load_data(init_file, ch1_file, ch2_file)


In [ ]:
# alice_key = MockBitBuffer(list(ch1))
# bob_key   = MockBitBuffer(list(ch2))


# #  initialize Alice and Bob
# alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
# bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
# #use same seed so they dont shuffle keys differently!!!!

# # start first pass
# alice_winnow.first_pass(permute_bits=True)
# bob_winnow.first_pass(permute_bits=True)

# # alice calculates syndrome for the first block 
# alice_syndrome = alice_winnow.get_syndrome(0)

# # bob uses Alice's syndrome to fix his key
# print(f"Bob's key before fix: {bob_winnow._key_string.bits[:8]}")
# bob_winnow.fix_with_syndrome(0, alice_syndrome)
# print(f"Bob's key after fix:  {bob_winnow._key_string.bits[:8]}")
# print(f"Alice's key: {alice_winnow._key_string.bits[:8]}")

# # verify that the keys match
# if alice_winnow._key_string.bits[:8] == bob_winnow._key_string.bits[:8]:
#     print(" Success! Bob corrected the error.")
# else:
#     print("Failure: Keys still do not match.")



In [ ]:
# alice_key = MockBitBuffer(list(ch1))
# bob_key   = MockBitBuffer(list(ch2))


# #  initialize Alice and Bob
# alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
# bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
# #use same seed so they dont shuffle keys differently!!!!

# # start first pass
# alice_winnow.first_pass()
# bob_winnow.first_pass()

# success_sample = 0
# total_sample = 0
# with open('corrected/cyrus_test.csv', 'w', newline='') as f:
#     # alice calculates syndrome for the first block 
#     for i in range(alice_winnow._num_of_blocks):
#         block_size = alice_winnow._block_size
#         startb = i*block_size
#         endb = block_size*(i + 1) 
#         alice_syndrome = alice_winnow.get_syndrome(i)
#         # bob uses Alice's syndrome to fix his key
#         if i % 1000 == 0:
#             print(f"Bob's key before fix: {bob_key.bits[startb:endb]} block {i}")
#         bob_winnow.fix_with_syndrome(i, alice_syndrome)
#         if i % 1000 == 0:
#             print(f"Bob's key after fix:  {bob_key.bits[startb:endb]} block {i}")
#             print(f"Alice's key: {alice_key.bits[startb:endb]}")

#         # verify that the keys match
#         if i % 100 == 0:
            
#             if alice_key.bits[startb:endb] == bob_key.bits[startb:endb]:
#                 print(f" Success! Bob corrected the error for block {i}.")
#                 success_sample+=1
#             else:
#                 print(f"Failure: Keys still do not match for block {i}.")
#             total_sample+=1
        
#         writer = csv.writer(f)
#         for item in bob_key.bits[:8]:
#             writer.writerow([item]) # Wrap item in a list

# if alice_key.bits == bob_key.bits:
#     print(" Success! Bob corrected the error.")
# else:
#     print("Failure: Keys still do not match.")

# print(f"Percentage success {(success_sample/total_sample )* 100} percent")

In [ ]:
# matches = (np.array(alice_key.bits)  == np.array(bob_key.bits) )

# # Calculate percentage
# percentage = matches.mean() * 100

# print(f"{percentage}%")

# Lets apply to many blocks!

In [ ]:
# print(alice_winnow._num_of_blocks)
# print(alice_winnow._block_size_schedule)
# alice_winnow.next_pass(permute_bits=True) 
# bob_winnow.next_pass(permute_bits=True) 
# print(alice_winnow._block_size_schedule)
# print(alice_winnow._num_of_blocks)

In [3]:
# THIS IS THE MAIN THING

def get_curr_qber():
    matches = (np.array(alice_winnow._key_string.bits)  != np.array(bob_winnow._key_string.bits) )

    # Calculate percentage
    percentage = matches.mean() * 100

    return percentage

alice_key = MockBitBuffer(list(ch1))
bob_key   = MockBitBuffer(list(ch2))


#  initialize Alice and Bob
alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
#use same seed so they dont shuffle keys differently!!!!

print(f"Qber before EC {get_curr_qber()}%")

# start first pass
alice_winnow.first_pass()
bob_winnow.first_pass()

success_sample = 0
total_sample = 0
with open('corrected/cyrus_test.csv', 'w', newline='') as f:
    # alice calculates syndrome for the first block 
    for i in range(alice_winnow._num_of_blocks):
        block_size = alice_winnow._block_size
        startb = i*block_size
        endb = block_size*(i + 1) 
        alice_syndrome = alice_winnow.get_syndrome(i)

        bob_winnow.fix_with_syndrome(i, alice_syndrome)

        

print(f"Qber after first pass {get_curr_qber()}%")

while alice_winnow.get_num_remaining_passes() > 0:
    # print("\nAlice internal:", alice_winnow._key_string.bits[:20])
    # print("Bob   internal:", bob_winnow._key_string.bits[:20])


    print(f"Alice seed: {alice_winnow._key_string.seed}")
    print(f"Bob seed:   {bob_winnow._key_string.seed}") 
    block_size  = alice_winnow._block_size
    num_blocks  = alice_winnow._num_of_blocks
    alice_winnow._key_string.discard_parity_bits(block_size, num_blocks)
    bob_winnow._key_string.discard_parity_bits(block_size, num_blocks)
    alice_winnow.next_pass(permute_bits=True) 
    bob_winnow.next_pass(permute_bits=True) 
    # print("Alice internal:", alice_winnow._key_string.bits[:20])
    # print("Bob   internal:", bob_winnow._key_string.bits[:20])
    # print(f"Alice pass number {alice_winnow._pass_number}")
    # print(f"Bob ass number {bob_winnow._pass_number}")
    # with open('corrected/cyrus_test.csv', 'w', newline='') as f:
    # # alice calculates syndrome for the first block 
    for i in range(alice_winnow._num_of_blocks):
        block_size = alice_winnow._block_size
        startb = i*block_size
        endb = block_size*(i + 1) 
        alice_syndrome = alice_winnow.get_syndrome(i)
        # print(f"Alice syndrome: {alice_winnow.get_syndrome(i)}")
        # print(f"Bob   syndrome: {bob_winnow.get_syndrome(i)}")
        # bob uses Alice's syndrome to fix his key
        # if i % 1000 == 0:
        #     print(f"Bob's key before fix: {bob_key.bits[startb:endb]} block {i}")
        bob_winnow.fix_with_syndrome(i, alice_syndrome)
        
        # if i % 1000 == 0:
        #     print(f"Bob's key after fix:  {bob_key.bits[startb:endb]} block {i}")
        #     print(f"Alice's key: {alice_key.bits[startb:endb]}")

        # # verify that the keys match
        # if i % 100 == 0:
            
        #     if alice_key.bits[startb:endb] == bob_key.bits[startb:endb]:
        #         print(f" Success! Bob corrected the error for block {i}.")
        #         # success_sample+=1
        #     else:
        #         print(f"Failure: Keys still do not match for block {i}.")
        #     # total_sample+=1
        
        # writer = csv.writer(f)
        # for item in bob_key.bits[:8]:
        #     writer.writerow([item]) # Wrap item in a list

    print(f"Qber {get_curr_qber()}%")



Qber before EC 24.382049560546875%


KeyboardInterrupt: 

In [3]:
#Lana testing
# THIS IS THE MAIN THING

def get_curr_qber():
    matches = (np.array(alice_winnow._key_string.bits)  != np.array(bob_winnow._key_string.bits) )

    # Calculate percentage
    percentage = matches.mean() * 100

    return percentage

alice_key = MockBitBuffer(list(ch1))
bob_key   = MockBitBuffer(list(ch2))


#  initialize Alice and Bob
alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
#use same seed so they dont shuffle keys differently!!!!

print(f"Qber before EC {get_curr_qber()}%")

# start first pass
alice_winnow.first_pass()
bob_winnow.first_pass()

success_sample = 0
total_sample = 0
with open('corrected/cyrus_test.csv', 'w', newline='') as f:
    # alice calculates syndrome for the first block 
    for i in range(alice_winnow._num_of_blocks):
        block_size = alice_winnow._block_size
        startb = i*block_size
        endb = block_size*(i + 1) 
        alice_syndrome = alice_winnow.get_syndrome(i)

        bob_winnow.fix_with_syndrome(i, alice_syndrome)

        

print(f"Qber after first pass {get_curr_qber()}%")

while alice_winnow.get_num_remaining_passes() > 0:
    # print("\nAlice internal:", alice_winnow._key_string.bits[:20])
    # print("Bob   internal:", bob_winnow._key_string.bits[:20])


    print(f"Alice seed: {alice_winnow._key_string.seed}")
    print(f"Bob seed:   {bob_winnow._key_string.seed}") 
    block_size  = alice_winnow._block_size
    num_blocks  = alice_winnow._num_of_blocks
    error_block_indices = list(range(num_blocks)) # Usually you discard from all blocks used
    alice_winnow.discard_syndrome_bits(error_block_indices)
    bob_winnow.discard_syndrome_bits(error_block_indices)
    alice_winnow.next_pass(permute_bits=True) 
    bob_winnow.next_pass(permute_bits=True) 
    # print("Alice internal:", alice_winnow._key_string.bits[:20])
    # print("Bob   internal:", bob_winnow._key_string.bits[:20])
    # print(f"Alice pass number {alice_winnow._pass_number}")
    # print(f"Bob ass number {bob_winnow._pass_number}")
    # with open('corrected/cyrus_test.csv', 'w', newline='') as f:
    # # alice calculates syndrome for the first block 
    for i in range(alice_winnow._num_of_blocks):
        block_size = alice_winnow._block_size
        startb = i*block_size
        endb = block_size*(i + 1) 
        alice_syndrome = alice_winnow.get_syndrome(i)
        # print(f"Alice syndrome: {alice_winnow.get_syndrome(i)}")
        # print(f"Bob   syndrome: {bob_winnow.get_syndrome(i)}")
        # bob uses Alice's syndrome to fix his key
        # if i % 1000 == 0:
        #     print(f"Bob's key before fix: {bob_key.bits[startb:endb]} block {i}")
        bob_winnow.fix_with_syndrome(i, alice_syndrome)
        
        # if i % 1000 == 0:
        #     print(f"Bob's key after fix:  {bob_key.bits[startb:endb]} block {i}")
        #     print(f"Alice's key: {alice_key.bits[startb:endb]}")

        # # verify that the keys match
        # if i % 100 == 0:
            
        #     if alice_key.bits[startb:endb] == bob_key.bits[startb:endb]:
        #         print(f" Success! Bob corrected the error for block {i}.")
        #         # success_sample+=1
        #     else:
        #         print(f"Failure: Keys still do not match for block {i}.")
        #     # total_sample+=1
        
        # writer = csv.writer(f)
        # for item in bob_key.bits[:8]:
        #     writer.writerow([item]) # Wrap item in a list

    print(f"Qber {get_curr_qber()}%")



Qber before EC 24.382049560546875%
Qber after first pass 17.305397033691406%
Alice seed: 42
Bob seed:   42
Qber 13.1004638671875%
Alice seed: 44
Bob seed:   44
Qber 10.44781494140625%
Alice seed: 47
Bob seed:   47
Qber 10.550048828125%
Alice seed: 51
Bob seed:   51
Qber 11.104225852272727%
Alice seed: 56
Bob seed:   56
Qber 13.086518595041321%


In [3]:
#Lana testing
# THIS IS THE MAIN THING

def get_curr_qber():
    matches = (np.array(alice_winnow._key_string.bits)  != np.array(bob_winnow._key_string.bits) )

    # Calculate percentage
    percentage = matches.mean() * 100

    return percentage

alice_key = MockBitBuffer(list(ch1))
bob_key   = MockBitBuffer(list(ch2))


#  initialize Alice and Bob
alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
#use same seed so they dont shuffle keys differently!!!!

print(f"Qber before EC {get_curr_qber()}%")

# start first pass
alice_winnow.first_pass()
bob_winnow.first_pass()

success_sample = 0
total_sample = 0
with open('corrected/cyrus_test.csv', 'w', newline='') as f:
    # alice calculates syndrome for the first block 
    for i in range(alice_winnow._num_of_blocks):
        block_size = alice_winnow._block_size
        startb = i*block_size
        endb = block_size*(i + 1) 
        alice_syndrome = alice_winnow.get_syndrome(i)

        bob_winnow.fix_with_syndrome(i, alice_syndrome)

        

print(f"Qber after first pass {get_curr_qber()}%")

while alice_winnow.get_num_remaining_passes() > 0:
    # print("\nAlice internal:", alice_winnow._key_string.bits[:20])
    # print("Bob   internal:", bob_winnow._key_string.bits[:20])


    print(f"Alice seed: {alice_winnow._key_string.seed}")
    print(f"Bob seed:   {bob_winnow._key_string.seed}") 
    block_size  = alice_winnow._block_size
    num_blocks  = alice_winnow._num_of_blocks
    error_block_indices = list(range(num_blocks)) # Usually you discard from all blocks used
    alice_winnow.discard_syndrome_bits(error_block_indices)
    bob_winnow.discard_syndrome_bits(error_block_indices)
    alice_winnow.next_pass(permute_bits=True) 
    bob_winnow.next_pass(permute_bits=True) 
    # print("Alice internal:", alice_winnow._key_string.bits[:20])
    # print("Bob   internal:", bob_winnow._key_string.bits[:20])
    # print(f"Alice pass number {alice_winnow._pass_number}")
    # print(f"Bob ass number {bob_winnow._pass_number}")
    # with open('corrected/cyrus_test.csv', 'w', newline='') as f:
    # # alice calculates syndrome for the first block 
    for i in range(alice_winnow._num_of_blocks):
        block_size = alice_winnow._block_size
        startb = i*block_size
        endb = block_size*(i + 1) 
        alice_syndrome = alice_winnow.get_syndrome(i)
        # print(f"Alice syndrome: {alice_winnow.get_syndrome(i)}")
        # print(f"Bob   syndrome: {bob_winnow.get_syndrome(i)}")
        # bob uses Alice's syndrome to fix his key
        # if i % 1000 == 0:
        #     print(f"Bob's key before fix: {bob_key.bits[startb:endb]} block {i}")
        bob_winnow.fix_with_syndrome(i, alice_syndrome)
        
        # if i % 1000 == 0:
        #     print(f"Bob's key after fix:  {bob_key.bits[startb:endb]} block {i}")
        #     print(f"Alice's key: {alice_key.bits[startb:endb]}")

        # # verify that the keys match
        # if i % 100 == 0:
            
        #     if alice_key.bits[startb:endb] == bob_key.bits[startb:endb]:
        #         print(f" Success! Bob corrected the error for block {i}.")
        #         # success_sample+=1
        #     else:
        #         print(f"Failure: Keys still do not match for block {i}.")
        #     # total_sample+=1
        
        # writer = csv.writer(f)
        # for item in bob_key.bits[:8]:
        #     writer.writerow([item]) # Wrap item in a list

    print(f"Qber {get_curr_qber()}%")



Qber before EC 24.382049560546875%
Qber after first pass 17.305397033691406%
Alice seed: 42
Bob seed:   42
Qber 13.1004638671875%
Alice seed: 44
Bob seed:   44
Qber 10.44781494140625%
Alice seed: 47
Bob seed:   47
Qber 7.2093505859375%
Alice seed: 51
Bob seed:   51
Qber 3.7570800781249996%
Alice seed: 56
Bob seed:   56
Qber 1.91357421875%
Alice seed: 62
Bob seed:   62
Qber 0.6995738636363636%
Alice seed: 69
Bob seed:   69
Qber 0.09710743801652892%
Alice seed: 77
Bob seed:   77
Qber 0.003005259203606311%


In [8]:

alice_winnow.discard_syndrome_bits(error_block_indices)
bob_winnow.discard_syndrome_bits(error_block_indices)

IndexError: list index out of range

In [4]:



# after all winnow passes and discards are done
alice_winnow._key_string.unpermute_all()
bob_winnow._key_string.unpermute_all()

# verify they still match after unscrambling

a = np.array(alice_winnow._key_string.bits)
b = np.array(bob_winnow._key_string.bits)
print(f"QBER after unscrambling: {(a != b).mean() * 100:.4f}%")

IndexError: list index out of range

# Now we can send a message between alice a

In [ ]:
import sys
import os

# Adds the parent directory to the search path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('winnow'), '..')))
from QCrypt_Methods import *

In [ ]:
ch1 = np.array(csv_to_int_array(ch1_file))
ch2 = np.array(csv_to_int_array(ch2_file))
keep_array = make_keep_array(ch1, ch2)
print(keep_array[30:100])

In [ ]:
# alice_str = "".join(str(n) for n in alice_winnow._key_string.bits)
# bob_str = "".join(str(n) for n in bob_winnow._key_string.bits)

fmat = bitstring_to_int_matrix(alice_winnow._key_string.bits)
plt.imshow(fmat/256, cmap=plt.get_cmap("gray"), aspect='auto')
plt.show()


In [ ]:
# # success_sample = 0
# # total_sample = 0
# with open('corrected/cyrus_test.csv', 'w', newline='') as f:
#     # alice calculates syndrome for the first block 
#     for i in range(alice_winnow._num_of_blocks):
#         block_size = alice_winnow._block_size
#         startb = i*block_size
#         endb = block_size*(i + 1) 
#         alice_syndrome = alice_winnow.get_syndrome(i)
#         # bob uses Alice's syndrome to fix his key
#         if i % 1000 == 0:
#             print(f"Bob's key before fix: {bob_key.bits[startb:endb]} block {i}")
#         bob_winnow.fix_with_syndrome(i, alice_syndrome)
#         if i % 1000 == 0:
#             print(f"Bob's key after fix:  {bob_key.bits[startb:endb]} block {i}")
#             print(f"Alice's key: {alice_key.bits[startb:endb]}")

#         # verify that the keys match
#         if i % 100 == 0:
            
#             if alice_key.bits[startb:endb] == bob_key.bits[startb:endb]:
#                 print(f" Success! Bob corrected the error for block {i}.")
#                 # success_sample+=1
#             else:
#                 print(f"Failure: Keys still do not match for block {i}.")
#             # total_sample+=1
        
#         writer = csv.writer(f)
#         for item in bob_key.bits[:8]:
#             writer.writerow([item]) # Wrap item in a list



# matches = (np.array(alice_key.bits)  == np.array(bob_key.bits) )

# # Calculate percentage
# percentage = matches.mean() * 100

# print(f"Qber is {percentage}%")

# if alice_key.bits == bob_key.bits:
#     print(" Success! Bob corrected the error.")
# else:
#     print("Failure: Keys still do not match.")

Lets do multiple passes!